In [1]:
# Import libraries and StackSats classes needed for exporting strategy weights, merging data, and plotting results.
import sys
import polars as pl
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

from stacksats.runner.core import StrategyRunner
from stacksats.strategy_types import ExportConfig
from stacksats.strategies.stable.baselines.uniform import UniformStrategy
from stacksats.strategies.stable.signals.momentum import MomentumStrategy

_root = Path.cwd()
while not (_root / "src").exists():
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from src import config as src_config, data_utils, plots, strategy_utils

STACKSATS_DATA_PATH = src_config.STACKSATS_DATA_PATH
RAW_PATH = src_config.RAW_PATH

data_utils.check_stacksats_data(STACKSATS_DATA_PATH, RAW_PATH)

True

In [2]:
# Initialize the runner and load the prepared Bitcoin analytics parquet manually.
runner = StrategyRunner()

btc_df = pl.read_parquet(STACKSATS_DATA_PATH).with_columns(pl.col("date").cast(pl.Datetime))

print("Loaded rows:", btc_df.height)
print(
    btc_df.select(
        pl.col("date").min().alias("min_date"),
        pl.col("date").max().alias("max_date")
    )
)

Loaded rows: 5689
shape: (1, 2)
┌─────────────────────┬─────────────────────┐
│ min_date            ┆ max_date            │
│ ---                 ┆ ---                 │
│ datetime[μs]        ┆ datetime[μs]        │
╞═════════════════════╪═════════════════════╡
│ 2010-08-16 00:00:00 ┆ 2026-03-13 00:00:00 │
└─────────────────────┴─────────────────────┘


In [3]:
btc_full = (
    btc_df
    .filter(
        (pl.col("date") >= pl.datetime(2010, 8, 16)) &
        (pl.col("date") <= pl.datetime(2023, 12, 31)) &
        pl.col("price_usd").is_not_null()
    )
    .sort("date")
)

print("Filtered rows:", btc_full.height)
print(
    btc_full.select(
        pl.col("date").min().alias("min_date"),
        pl.col("date").max().alias("max_date")
    )
)

Filtered rows: 4886
shape: (1, 2)
┌─────────────────────┬─────────────────────┐
│ min_date            ┆ max_date            │
│ ---                 ┆ ---                 │
│ datetime[μs]        ┆ datetime[μs]        │
╞═════════════════════╪═════════════════════╡
│ 2010-08-16 00:00:00 ┆ 2023-12-31 00:00:00 │
└─────────────────────┴─────────────────────┘


In [4]:
cycle_results = {}

for cycle in plots.calendar_cycles:
    result = strategy_utils.process_cycle_year_by_year(
        cycle=cycle,
        btc_data=btc_full,
        runner=runner,
        dynamic_strategy=MomentumStrategy(),
        total_budget_usd=src_config.TOTAL_BUDGET_USD,
        top_buy_quantile=src_config.TOP_BUY_QUANTILE,
    )

    cycle_results[cycle["label"]] = result


Processing Cycle 1: 2010-2013
shape: (1, 3)
┌─────────────────────┬─────────────────────┬──────┐
│ min_date            ┆ max_date            ┆ rows │
│ ---                 ┆ ---                 ┆ ---  │
│ datetime[μs]        ┆ datetime[μs]        ┆ u32  │
╞═════════════════════╪═════════════════════╪══════╡
│ 2010-08-16 00:00:00 ┆ 2013-12-31 00:00:00 ┆ 1234 │
└─────────────────────┴─────────────────────┴──────┘
2010: skipped, less than 365 rows
2010: skipped, less than 365 rows


2011: exported 365 rows
2011: exported 365 rows
2012: exported 365 rows


2012: exported 365 rows
2013: exported 365 rows
2013: exported 365 rows

Processing Cycle 2: 2014-2017
shape: (1, 3)
┌─────────────────────┬─────────────────────┬──────┐
│ min_date            ┆ max_date            ┆ rows │
│ ---                 ┆ ---                 ┆ ---  │
│ datetime[μs]        ┆ datetime[μs]        ┆ u32  │
╞═════════════════════╪═════════════════════╪══════╡
│ 2014-01-01 00:00:00 ┆ 2017-12-31 00:00:00 ┆ 1461 │
└─────────────────────┴─────────────────────┴──────┘


2014: exported 365 rows
2014: exported 365 rows


2015: exported 365 rows
2015: exported 365 rows


...

2020: exported 365 rows


2020: exported 365 rows
2021: exported 365 rows


2021: exported 365 rows

Processing Cycle 4: 2022-2023
shape: (1, 3)
┌─────────────────────┬─────────────────────┬──────┐
│ min_date            ┆ max_date            ┆ rows │
│ ---                 ┆ ---                 ┆ ---  │
│ datetime[μs]        ┆ datetime[μs]        ┆ u32  │
╞═════════════════════╪═════════════════════╪══════╡
│ 2022-01-01 00:00:00 ┆ 2023-12-31 00:00:00 ┆ 730  │
└─────────────────────┴─────────────────────┴──────┘
2022: exported 365 rows


2022: exported 365 rows
2023: exported 365 rows


2023: exported 365 rows


In [5]:
cols = plots.StrategyColumns(
    weight="dynamic_weight",
    spd="sats_per_dollar_dynamic",
    sats_accum="sats_accum_dynamic",
)

# Combine all 4 cycles into one DataFrame
combined_plot_df = pd.concat(
    [r["plot_df"] for r in cycle_results.values()]
).sort_values("date").reset_index(drop=True)

# per cycle plots
cycle_plots = plots.plot_strategy_by_cycle(combined_plot_df, cols, "Momentum")
plt.show()

In [6]:
# Build monthly table for each cycle.
def build_monthly_table(result: dict):
    merged = result["merged"]

    monthly = (
        merged
        .with_columns(pl.col("date").dt.truncate("1mo").alias("month"))
        .group_by("month")
        .agg([
            pl.col("price_usd").mean().alias("avg_price_usd"),
            pl.col("dynamic_weight_raw").sum().alias("total_dynamic_weight"),
            pl.col("baseline_weight_raw").sum().alias("total_baseline_weight"),
            pl.col("sats_accum_dynamic").sum().alias("sats_accum_dynamic"),
            pl.col("sats_accum_baseline").sum().alias("sats_accum_baseline"),
            pl.col("dynamic_usd").sum().alias("total_dynamic_usd"),
            pl.col("baseline_usd").sum().alias("total_baseline_usd"),
        ])
        .with_columns([
            (pl.col("sats_accum_dynamic") / pl.col("total_dynamic_usd")).alias("avg_sats_per_dollar_dynamic"),
            (pl.col("sats_accum_baseline") / pl.col("total_baseline_usd")).alias("avg_sats_per_dollar_baseline"),
        ])
        .sort("month")
    )

    monthly_pd = monthly.to_pandas()
    monthly_pd["month"] = monthly_pd["month"].dt.strftime("%Y-%m")
    monthly_pd["avg_price_usd"] = monthly_pd["avg_price_usd"].round(2)

    for col in [
        "avg_sats_per_dollar_dynamic",
        "avg_sats_per_dollar_baseline",
        "total_dynamic_weight",
        "total_baseline_weight",
        "sats_accum_dynamic",
        "sats_accum_baseline",
    ]:
        monthly_pd[col] = monthly_pd[col].round(4)

    monthly_pd = monthly_pd[[
        "month",
        "avg_price_usd",
        "avg_sats_per_dollar_dynamic",
        "avg_sats_per_dollar_baseline",
        "total_dynamic_weight",
        "total_baseline_weight",
        "sats_accum_dynamic",
        "sats_accum_baseline",
    ]]

    return monthly_pd

In [7]:
# Create monthly tables for all 4 cycles.
cycle_monthly_tables = {
    label: build_monthly_table(result)
    for label, result in cycle_results.items()
}

In [8]:
# per year plots
year_plots = plots.plot_strategy_by_year(combined_plot_df, cols, "Momentum")
plt.show()

In [9]:
cycle1 = cycle_results["Cycle 1: 2010-2013"]["merged"]

(
    cycle1
    .sort("dynamic_weight", descending=True)
    .head(20)
    .select(["date", "price_usd", "dynamic_weight", "baseline_weight"])
    .to_pandas()
)

,date,price_usd,dynamic_weight,baseline_weight
0,2012-12-31,13.24,0.060400,0.00274
1,2013-05-09,111.97,0.007237,0.00274
2,2011-07-09,14.39,0.006703,0.00274
3,2013-05-08,113.47,0.006498,0.00274
4,2011-07-08,14.35,0.006489,0.00274
5,2013-05-03,94.26,0.006160,0.00274
6,2013-05-07,111.43,0.006131,0.00274
7,2011-07-10,15.08,0.005881,0.00274
8,2011-08-06,7.82,0.005877,0.00274
9,2011-08-07,7.72,0.005719,0.00274


In [10]:
cycle1 = cycle_results["Cycle 1: 2010-2013"]["merged"]

(
    cycle1
    .filter(pl.col("date").dt.year() == 2011)
    .sort("dynamic_weight", descending=True)
    .head(20)
    .select(["date", "price_usd", "dynamic_weight", "baseline_weight"])
    .to_pandas()
)

,date,price_usd,dynamic_weight,baseline_weight
0,2011-07-09,14.39,0.006703,0.00274
1,2011-07-08,14.35,0.006489,0.00274
2,2011-07-10,15.08,0.005881,0.00274
3,2011-08-06,7.82,0.005877,0.00274
4,2011-08-07,7.72,0.005719,0.00274
5,2011-08-08,7.74,0.005692,0.00274
6,2011-09-17,4.77,0.005620,0.00274
7,2011-09-16,4.81,0.005617,0.00274
8,2011-09-15,4.97,0.005597,0.00274
9,2011-09-18,5.11,0.005538,0.00274


In [11]:
(
    cycle1
    .filter(pl.col("date") == pl.datetime(2011, 12, 31))
    .select(["date", "price_usd", "dynamic_weight", "baseline_weight"])
    .to_pandas()
)

,date,price_usd,dynamic_weight,baseline_weight
0,2011-12-31,4.58,0.00001,0.00274


In [12]:
cycle1 = cycle_results["Cycle 1: 2010-2013"]["merged"]

(
    cycle1
    .filter(pl.col("date").dt.year() == 2012)
    .sort("dynamic_weight", descending=True)
    .head(20)
    .select(["date", "price_usd", "dynamic_weight", "baseline_weight"])
    .to_pandas()
)

,date,price_usd,dynamic_weight,baseline_weight
0,2012-12-31,13.24,0.060400,0.00274
1,2012-02-19,4.38,0.003717,0.00274
2,2012-02-15,4.69,0.003702,0.00274
3,2012-02-17,4.67,0.003667,0.00274
4,2012-02-14,4.89,0.003646,0.00274
5,2012-02-20,4.44,0.003600,0.00274
6,2012-02-18,4.23,0.003563,0.00274
7,2012-10-26,9.88,0.003536,0.00274
8,2012-02-21,4.58,0.003513,0.00274
9,2012-11-02,10.52,0.003499,0.00274


In [13]:
(
    cycle1
    .filter(pl.col("date") == pl.datetime(2012, 12, 31))
    .select(["date", "price_usd", "dynamic_weight", "baseline_weight"])
    .to_pandas()
)

,date,price_usd,dynamic_weight,baseline_weight
0,2012-12-31,13.24,0.0604,0.00274


In [14]:
cycle1_2012 = (
    cycle1
    .filter(pl.col("date").dt.year() == 2012)
    .sort("date")
    .with_columns([
        pl.col("dynamic_weight").cum_sum().alias("cum_dynamic_weight"),
        pl.col("baseline_weight").cum_sum().alias("cum_baseline_weight")
    ])
)

cycle1_2012.tail(10).to_pandas()

,date,price_usd,dynamic_weight_raw,baseline_weight_raw,year,dynamic_weight,baseline_weight,dynamic_usd,baseline_usd,btc_accum_dynamic,btc_accum_baseline,sats_accum_dynamic,sats_accum_baseline,sats_per_dollar_dynamic,sats_per_dollar_baseline,cum_dynamic_weight,cum_baseline_weight
0,2012-12-22,13.20,0.002609,0.00274,2012,0.002609,0.00274,2.608598,2.739726,0.197621,0.207555,1.976211e+07,2.075550e+07,7.575758e+06,7.575758e+06,0.917993,0.975342
1,2012-12-23,13.14,0.002682,0.00274,2012,0.002682,0.00274,2.682417,2.739726,0.204141,0.208503,2.041413e+07,2.085027e+07,7.610350e+06,7.610350e+06,0.920676,0.978082
2,2012-12-24,13.23,0.002691,0.00274,2012,0.002691,0.00274,2.691322,2.739726,0.203426,0.207084,2.034257e+07,2.070844e+07,7.558579e+06,7.558579e+06,0.923367,0.980822
3,2012-12-25,13.24,0.002703,0.00274,2012,0.002703,0.00274,2.703429,2.739726,0.204187,0.206928,2.041865e+07,2.069279e+07,7.552870e+06,7.552870e+06,0.926071,0.983562
4,2012-12-26,13.18,0.002691,0.00274,2012,0.002691,0.00274,2.690673,2.739726,0.204148,0.207870,2.041482e+07,2.078700e+07,7.587253e+06,7.587253e+06,0.928761,0.986301
5,2012-12-27,13.20,0.002655,0.00274,2012,0.002655,0.00274,2.655371,2.739726,0.201164,0.207555,2.011645e+07,2.075550e+07,7.575758e+06,7.575758e+06,0.931417,0.989041
6,2012-12-28,13.18,0.002696,0.00274,2012,0.002696,0.00274,2.695785,2.739726,0.204536,0.207870,2.045361e+07,2.078700e+07,7.587253e+06,7.587253e+06,0.934112,0.991781
7,2012-12-29,13.11,0.002747,0.00274,2012,0.002747,0.00274,2.746754,2.739726,0.209516,0.208980,2.095160e+07,2.089799e+07,7.627765e+06,7.627765e+06,0.936859,0.994521
8,2012-12-30,13.20,0.002741,0.00274,2012,0.002741,0.00274,2.740921,2.739726,0.207646,0.207555,2.076455e+07,2.075550e+07,7.575758e+06,7.575758e+06,0.939600,0.997260
9,2012-12-31,13.24,0.060400,0.00274,2012,0.060400,0.00274,60.399840,2.739726,4.561921,0.206928,4.561921e+08,2.069279e+07,7.552870e+06,7.552870e+06,1.000000,1.000000


In [15]:
(
    cycle1_2012
    .filter(pl.col("date") < pl.datetime(2012, 12, 31))
    .select([
        pl.col("dynamic_weight").sum().alias("dynamic_sum_before_last_day"),
        pl.col("baseline_weight").sum().alias("baseline_sum_before_last_day")
    ])
    .to_pandas()
)

,dynamic_sum_before_last_day,baseline_sum_before_last_day
0,0.9396,0.99726


In [16]:
cycle1_2011 = (
    cycle1
    .filter(pl.col("date").dt.year() == 2011)
    .sort("date")
    .with_columns([
        pl.col("dynamic_weight").cum_sum().alias("cum_dynamic_weight"),
        pl.col("baseline_weight").cum_sum().alias("cum_baseline_weight")
    ])
)

cycle1_2011.tail(10).to_pandas()

,date,price_usd,dynamic_weight_raw,baseline_weight_raw,year,dynamic_weight,baseline_weight,dynamic_usd,baseline_usd,btc_accum_dynamic,btc_accum_baseline,sats_accum_dynamic,sats_accum_baseline,sats_per_dollar_dynamic,sats_per_dollar_baseline,cum_dynamic_weight,cum_baseline_weight
0,2011-12-22,3.79,0.00001,0.00274,2011,0.00001,0.00274,0.01,2.739726,0.002639,0.722883,263852.242762,7.228829e+07,2.638522e+07,2.638522e+07,0.99991,0.975342
1,2011-12-23,3.90,0.00001,0.00274,2011,0.00001,0.00274,0.01,2.739726,0.002564,0.702494,256410.256419,7.024939e+07,2.564103e+07,2.564103e+07,0.99992,0.978082
2,2011-12-24,3.92,0.00001,0.00274,2011,0.00001,0.00274,0.01,2.739726,0.002551,0.698910,255102.040823,6.989097e+07,2.551020e+07,2.551020e+07,0.99993,0.980822
3,2011-12-25,4.14,0.00001,0.00274,2011,0.00001,0.00274,0.01,2.739726,0.002415,0.661770,241545.893783,6.617696e+07,2.415459e+07,2.415459e+07,0.99994,0.983562
4,2011-12-26,4.04,0.00001,0.00274,2011,0.00001,0.00274,0.01,2.739726,0.002475,0.678150,247524.752549,6.781500e+07,2.475248e+07,2.475248e+07,0.99995,0.986301
5,2011-12-27,4.02,0.00001,0.00274,2011,0.00001,0.00274,0.01,2.739726,0.002488,0.681524,248756.218982,6.815239e+07,2.487562e+07,2.487562e+07,0.99996,0.989041
6,2011-12-28,4.14,0.00001,0.00274,2011,0.00001,0.00274,0.01,2.739726,0.002415,0.661770,241545.893783,6.617696e+07,2.415459e+07,2.415459e+07,0.99997,0.991781
7,2011-12-29,4.22,0.00001,0.00274,2011,0.00001,0.00274,0.01,2.739726,0.002370,0.649224,236966.824699,6.492242e+07,2.369668e+07,2.369668e+07,0.99998,0.994521
8,2011-12-30,4.19,0.00001,0.00274,2011,0.00001,0.00274,0.01,2.739726,0.002387,0.653873,238663.484545,6.538726e+07,2.386635e+07,2.386635e+07,0.99999,0.997260
9,2011-12-31,4.58,0.00001,0.00274,2011,0.00001,0.00274,0.01,2.739726,0.002183,0.598193,218340.611379,5.981935e+07,2.183406e+07,2.183406e+07,1.00000,1.000000


In [17]:
(
    cycle1_2011
    .filter(pl.col("date") < pl.datetime(2012, 12, 31))
    .select([
        pl.col("dynamic_weight").sum().alias("dynamic_sum_before_last_day"),
        pl.col("baseline_weight").sum().alias("baseline_sum_before_last_day")
    ])
    .to_pandas()
)

,dynamic_sum_before_last_day,baseline_sum_before_last_day
0,1.0,1.0
